# Preprocessing

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

## Data Loading

In [ ]:
# Load dataframe
df = pd.read_csv("../data/raw/listings.csv")

Based on domain-knowledge, I load the relevant columns from the more detailed dataset. Location and room type are typical price drivers for accommodations. Higher-quality listings tend to cost more, and quality is reflected in the ratings. Price also increases with capacity, captured by the number of guests it accommodates and the number of bedrooms. Finally, convenience features like instant booking and superhost status may justify a higher price.

The information on the number of beds is not selected, because it is unreliable on AirBNB, as the definition of a bed is ambiguous. `accommodates` captures the sleeping capacity more reliably.

In [ ]:
# Load relevant columns of the more detailed dataframe, based on domain-knowledge
detail_ratings = ["review_scores_cleanliness", "review_scores_checkin", 
               "review_scores_communication", "review_scores_location", 
               "review_scores_accuracy", "review_scores_value"]

add_df = pd.read_csv(
    "../data/raw/listings.csv.gz",
    usecols=["id"] + detail_ratings + ["review_scores_rating", "host_is_superhost", "bedrooms", "accommodates", "instant_bookable"]
)

In [ ]:
# Merge both dataframes
df = df.merge(
    add_df,
    on="id",
    how="left"
)
df.head(3)

## Data Cleaning

### Column Selection

`neighbourhood_group` has no existing values, therefore remove it.

In [ ]:
not_null_count = df["neighbourhood_group"].notna().sum()
print(f"Number of existing values: {not_null_count}")

# Remove column
df = df.drop(columns=["neighbourhood_group"])

Remove columns with no useful information on the price (except the id):

In [ ]:
df = df.drop(columns=["name", "host_id", "host_name", "license"])

Remove columns based on domain knowledge:
- `minimum_nights` and `availability_365` describe booking constraints rather than properties of the listing itself.
- `last_review` is a date that has little direct connection to the price.
- `calculated_host_listings_count` reflects the host rather than the individual listing.

In [ ]:
df = df.drop(columns = ["minimum_nights", "last_review", "calculated_host_listings_count", "availability_365"])

`reviews_per_month` is redundant because of `number_of_reviews`, therefore remove it.

In [ ]:
df = df.drop(columns="reviews_per_month")

When the overall rating `review_scores_rating` is available, then all the more detailed ratings are also available:

In [ ]:
mask = df["review_scores_rating"].notna()
print("Number of missing values of detailed ratings, where overall rating exists:")
print(df.loc[mask, detail_ratings].isna().sum())

The overall rating `review_scores_rating` includes the price-performance ratio `review_scores_value`, which is directly influenced by price and would introduce reverse causality.

Because of the completeness of the more detailed ratings, I choose to use the average of all detailed ratings, excluding `review_scores_value`, instead of `review_scores_rating`:

In [ ]:
# Compute custom rating
rating_cols = [col for col in detail_ratings if col != "review_scores_value"]
df["avg_rating"] = df[rating_cols].mean(axis=1)

# Remove other ratings
df = df.drop(columns=detail_ratings+["review_scores_rating"])

Looking at missing values of the rating, we can observe that all listings without review have a missing rating. They are imputed with 0 and the binary variable `has_reviews` is added to indicate the absence of reviews rather than a low rating.
 
Both variables will be included in the linear regression model.

In [ ]:
# Compute number of listings without review, that have a rating
num = df.loc[df["number_of_reviews"]==0, "avg_rating"].notna().sum()
print(f"{num} listings with no review have a rating.") 

# Creating indicator variable
df["has_reviews"] = (df["number_of_reviews"]>0)

# Imputing null-values with 0
df.loc[~df["has_reviews"], "avg_rating"] = 0

`number_of_reviews_ltm` is the number of reviews in the last twelve months and is more current than the total `number_of_reviews`, which is therefore removed.

In [ ]:
# Drop `number_of_reviews`
df = df.drop(columns=["number_of_reviews"])

`neighbourhood` captures geographic information of listings and makes `longitude` and `latitude` quite redundant, as shown in the scatter plot below. 

I rather use this variable than to compute the distance to the city center, because neighbourhood boundaries likely reflect price differences better than raw distance. Therefore, `longitude` and `latitude` are removed.

In [ ]:
# Plot neighbourhoods by geographic location
sns.scatterplot(
    x="longitude", 
    y="latitude", 
    hue="neighbourhood",
    data=df,
    s=10
)
plt.title("Neighbourhoods of Florence")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.show()

# Remove geographic information of listings
df = df.drop(columns=["longitude", "latitude"])

Reorder columns

In [ ]:
# Reorder dataframe columns into a meaningful order
df = df[["id", "neighbourhood", "room_type", "accommodates", 
         "bedrooms", "host_is_superhost", "instant_bookable",
         "number_of_reviews_ltm", "has_reviews", "avg_rating",
         "price"]]
df.head(3)

### Data Type Conversion

In [ ]:
# Convert best possible Pandas dtypes
df = df.convert_dtypes()

# `host_is_superhost` and `instant_bookable` as boolean
bool_map = {"t": True, "f": False}
df["host_is_superhost"] = df["host_is_superhost"].map(bool_map).astype("boolean")
df["instant_bookable"] = df["instant_bookable"].map(bool_map).astype("boolean")

# String to category
df["neighbourhood"] = df["neighbourhood"].astype("category")
df["room_type"] = df["room_type"].astype("category")

df.head(3)

### Target Variable (Price)

#### Missing Values

Around 9% of price values are missing. A comparison of the feature-distributions showed only small differences and no strong indication of systematic bias.

Because imputing a target variable is not sensible, the rows with missing price values are dropped.

In [ ]:
missing_pct = df["price"].isna().mean()*100
print(f"Missing price values: {missing_pct:.1f}%")

# Remove rows with missing price values
df = df.dropna(subset=["price"])

#### Outliers

In [ ]:
quantiles = np.arange(0.93, 1.0, 0.005)
values = df["price"].quantile(quantiles)
plt.plot(quantiles, values)
plt.xlabel("Quantile")
plt.ylabel("Price")
plt.title("Price distribution at high quantiles")
plt.show()

Listings above the 98.5th percentile are removed, because the quantile plot shows a steep increase after this point, indicating luxury properties that are not representative of typical Airbnb listings in Florence.

In [ ]:
q985 = df["price"].quantile(0.985)
df = df[df["price"]<=q985]

For economic target variables like `price`, the underlying relations are often multiplicative, which leads to a right-skewed distribution in the data. Applying the logarithm transforms these into additive relations. The histogram below confirms the target-variable `price` is right-skewed. 

The log-transformation also reduces the impact of outliers, which is particularly important for the target variable because linear regression minimizes the squared residuals of the target variable.

It also makes the distribution more symmetric, which makes it more likely to meet the regression assumptions of normally distributed residuals. These assumptions will still be verified after fitting the model.

For these reasons, I will use `log_price` as the target variable during the exploration and analysis.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
df["log_price"] = np.log(df["price"])

sns.histplot(df["price"], bins=45, kde=True, ax=ax[0])
ax[0].set_xlabel("Price")
ax[0].set_ylabel("Count")
ax[0].set_title("Target variable (Price) is right-skewed")

sns.histplot(df["log_price"], bins=45, kde=True, ax=ax[1])
ax[1].set_xlabel("Log(Price)")
ax[1].set_ylabel("Count")
ax[1].set_title("Log(Price) has a more symmetric distribution")

plt.tight_layout()
plt.show()

### Handling Outliers

All values of numerical variables appear realistic, as can be seen in the table below.

Influential outliers will be identified after fitting the initial linear regression model using Cook's Distance and removed if neccessary.

In [ ]:
df[["accommodates", "bedrooms", "number_of_reviews_ltm", "avg_rating"]].describe()

### Handling Missing Values

Following columns have missing values:

In [ ]:
# Percentage of missing values
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
pd.DataFrame({
    'missing_count': missing,
    'missing_pct': (missing / len(df) * 100).round(2)
})

The missing values of `bedrooms`, `accommodates`, `instant_bookable` and `avg_rating` are all within the same 11 rows.

These rows are dropped, because the information loss is negligible and a look at these rows doesn't show an obvious pattern - most listings are entire homes or apartments located in Centro Storico, but these two categories do dominate in this dataset anyway.

In [ ]:
# Show rows with common missing values among features with less than 0.1% missing values
cols = ["bedrooms", "accommodates", "instant_bookable", "avg_rating"]
display(df.loc[df[cols].isna().any(axis=1), ["neighbourhood", "room_type", "number_of_reviews_ltm", "price"]+cols])

# Compute percentage of dominating categories
pct1 = (df["neighbourhood"]=="Centro Storico").mean()*100
print(f"{pct1:.1f}% of listings are in Centro Storico")
pct2 = (df["room_type"]=="Entire home/apt").mean()*100
print(f"{pct2:.1f}% of listings are entire homes or apartments")

# Drop missing values of those rows
len_before = len(df)
df = df.dropna(subset=["bedrooms", "accommodates", "instant_bookable", "avg_rating"])
len_after = len(df)
print(f"\n{len_before-len_after} rows have been dropped.")

#### Missing values: host_is_superhost
I further analyse `host_is_superhost` to find out whether the missingness can be assumed MCAR, or whether MAR can be shown. MNAR depends on the unobserved values themselves and cannot be tested from the data alone, therefore my analysis distinguishes between MAR and MCAR.

To detect possible MAR, the missingness of `host_is_superhost` is modelled as a binary target variable and it is checked whether the other variables can predict it. I use logistic regression because it gives interpretable coefficients with p-values.

##### Assumptions
The assumptions of logistic regression are...

... for valid coefficient estimates:
- **Linear Decision Boundary.** Checked after the regression - since MAR was detected, this assumption has no relevance because a too simple boundary can only miss relations but not create them.
- **No (near-)perfect separation.** The maximum likelihood optimizer does not converge when a predictor perfectly (or almost perfectly) separates the target. This is reported by the model itself with a convergence warning (see below).

... for valid standard errors:
- **Conditional independence of observations given the predictors.** In the case of Airbnb, there is one violation of this assumption: Multiple listings per host - this is handled with cluster-robust standard errors using `host_id`, which I will retrieve from the raw data. The local dependency of Airbnb listings is largely captured by the `neighbourhood` predictor.

In [ ]:
# Add information of host id
df_raw = pd.read_csv("../data/raw/listings.csv")
print(f"{len(df)} rows before merge")
df = df.merge(df_raw[["id", "host_id"]], on="id", how="left").convert_dtypes()
print(f"{len(df)} rows after merge")

The dataset indeed contains many hosts with multiple listings:

In [ ]:
# Computing statistics about the host-clusters
counts = df["host_id"].value_counts()
print(f"The dataset contains: {df['host_id'].nunique()} hosts, {len(df)} listings")
print(f"Hosts with more than one listing: {(counts > 1).mean():.1%}")
print(f"Most listings of a single host: {counts.max()} listings")

Logistic regression works only on numerical data, therefore categorical features are one-hot-encoded, dropping the first category to avoid collinearity.

The target is unbalanced, therefore AUC is used as the metric instead of accuracy.

In [ ]:
from sklearn.metrics import roc_auc_score

# Create the target
df_model = df.copy()
df_model["superhost_missing"] = df_model["host_is_superhost"].isna().astype(int)
y = df_model["superhost_missing"]

# Predictors: All features but the tested feature, the target, the ids and the price
drop_cols = ["host_is_superhost", "superhost_missing", "price",
             "id", "host_id"]
X = df_model.drop(columns=drop_cols)

# One-hot-encoding of categorical data
categ = X.select_dtypes("category").columns
X = pd.get_dummies(X, columns=list(categ), drop_first=True)

# Ensure that numerical data is used (especially casting booleans to floats)
X = X.astype(float)

# Add intercept
X = sm.add_constant(X)

# Logistic regression with cluster-robust standard errors on 'host_id'
model = sm.Logit(y, X).fit(
    disp=False, cov_type="cluster", cov_kwds={"groups": df_model["host_id"]}
)

# Check the model's performance - a value of 0.5 means random guess
auc = roc_auc_score(y, model.predict(X))
print(f"AUC: {auc:.3f}")

# Check for significant coefficients (having a p-value below 0.05)
sig_features = model.pvalues[model.pvalues < 0.05].drop("const", errors="ignore")
if sig_features.empty:
    print("No significant feature. The test is consistent with MCAR missingness.")
else:
    print("Significant features related to the missingness and their p-value:")
    print(sig_features.to_string())

The maximum likelihood optimizer did not converge, which is likely due to perfect separation. 

A look at the features shows that there is indeed perfect separation, namely in the category "Shared room":

In [ ]:
pd.crosstab(df_model["room_type"], df_model["superhost_missing"])

Since our model found a second significant variable (`instant_bookable`), the feature `room_type` is excluded and it is checked if the variable is still significant:

In [ ]:
# Remove the dummies of the categorical variable `room_type`
X = X.drop(columns=[c for c in X.columns if c.startswith("room_type_")])

# Logistic regression with cluster-robust standard errors to `host_id`
model = sm.Logit(y, X).fit(
    disp=False, cov_type="cluster", cov_kwds={"groups": df_model["host_id"]}
)

# Check the model's performance - a value of 0.5 means random guess
auc = roc_auc_score(y, model.predict(X))
print(f"AUC: {auc:.3f}")

# Check for significant coefficients (having a p-value below 0.05)
sig_features = model.pvalues[model.pvalues < 0.05].drop("const", errors="ignore")
if sig_features.empty:
    print("No significant feature. The test is consistent with MCAR missingness.")
else:
    print("Significant features related to the missingness and their p-value:")
    print(sig_features.to_string())

The model still reports `instant_bookable` as significant.
Looking at the missingness rate per category confirms the relation: it is more than three times as high for instant-bookable listings (10.1% vs. 3.1%).

In [ ]:
# Percentage of missing superhost values
missing_pct = (df_model.groupby("instant_bookable")["superhost_missing"].mean()*100).round(1)
pd.DataFrame({
    'missing_pct': missing_pct,
})

With that I conclude that the missingness is MAR and contains information itself. Therefore I handle it by creating a 3rd category "missing" of the feature `host_is_superhost`.

In [ ]:
# Add new category "missing" and fill null values
# Cast to string first, so all categories share one dtype
df["host_is_superhost"] = df["host_is_superhost"].astype("string").astype("category")
df["host_is_superhost"] = (
    df["host_is_superhost"]
    .cat.add_categories("missing")
    .fillna("missing")
)
df.head(3)

## Save Preprocessed Data to File

In [ ]:
df.to_parquet("../data/processed/listings_processed.parquet", index=False)